# `HR.TRG_EMP` Full Load

**Conversion Timestamp:** 2024-07-30T12:00:00Z

This notebook performs a full load into `workspace.hr.trg_emp` from `workspace.hr.employees`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
display(spark.sql("""
SELECT
  '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
  ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,
  ${ETL_PROC_WID} AS ETL_PROC_WID,
  '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## Data Load

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}

INSERT INTO workspace.hr.trg_emp
  (
    employee_id ,
    first_name ,
    last_name ,
    email ,
    phone_number ,
    hire_date ,
    job_id ,
    salary ,
    commission_pct ,
    manager_id ,
    department_id
  )
SELECT
  employees.employee_id ,
  employees.first_name ,
  employees.last_name ,
  employees.email ,
  employees.phone_number ,
  employees.hire_date ,
  employees.job_id ,
  employees.salary ,
  employees.commission_pct ,
  employees.manager_id ,
  employees.department_id
FROM
  workspace.hr.employees AS employees;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_target FROM workspace.hr.trg_emp;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** Original Oracle schema `HR` has been converted to `workspace.hr`. Table names `TRG_EMP` and `EMPLOYEES` have been converted to lowercase `trg_emp` and `employees` respectively.
2.  **Oracle Hint Removal:** The Oracle-specific hint `/*+ APPEND PARALLEL */` has been removed as it is not applicable in Databricks Delta Lake.
3.  **ODI SCEN_TASK_NO:** The original `SCEN_TASK_NO` directives are preserved as comments within the respective SQL cells.
4.  **Data Types:** Implicit conversion of Oracle data types (`NUMBER`, `VARCHAR2`, `DATE`) to Spark SQL equivalents (`BIGINT`/`DECIMAL`, `STRING`, `TIMESTAMP`) is assumed based on existing target table DDL. If `workspace.hr.trg_emp` does not exist or has incorrect data types, it should be created or altered with appropriate Spark SQL types.
5.  **Error Handling & Logging:** This conversion only covers the DML statement. Original ODI error handling and logging (e.g., `E$` tables, `SNP_CHECK_TAB`) are not present in the source snippet and thus not included. Databricks-native error logging mechanisms should be implemented if required.
6.  **Full Load Behavior:** This `INSERT` statement performs a full load. If the target table `workspace.hr.trg_emp` is not empty, this will add new rows without clearing existing data. If a TRUNCATE/DELETE + INSERT behavior is desired for a true full load, a `TRUNCATE TABLE workspace.hr.trg_emp;` or `DELETE FROM workspace.hr.trg_emp;` statement should precede the `INSERT`.